In [2]:
# Supervisor Pattern cơ bản
import os
from typing import Annotated, TypedDict, Literal
from pydantic import BaseModel
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(override=True)

True

In [1]:
from pydantic import BaseModel, Field
from typing import Optional

class KetQuaDanhGia(BaseModel):
    """Kết quả đánh giá của Evaluator."""
    da_dat_tieu_chuan: bool = Field(
        description="True nếu output của Worker đã đạt tiêu chuẩn"
    )
    can_them_thong_tin_nguoi_dung: bool = Field(
        description="True nếu Worker bị kẹt và cần người dùng cung cấp thêm thông tin"
    )
    phan_hoi_chi_tiet: str = Field(
        description="Nhận xét cụ thể: điểm mạnh, điểm yếu, cần cải thiện gì"
    )
    diem_chat_luong: int = Field(
        description="Điểm chất lượng từ 1-10",
        ge=1, le=10
    )

In [3]:
# State
class TrangThaiSanXuat(TypedDict):
    messages: Annotated[list, add_messages]
    yeu_cau_goc: str
    tieu_chuan_thanh_cong: str

    output_hien_tai: str
    phan_hoi_evaluator: str
    da_hoan_thanh: bool
    can_nguoi_dung: bool
    so_vong_thu: int
    diem_tot_nhat: int
    

In [4]:
# Node Worker & Node Evaluator
llm_worker = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

llm_evaluator = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1
).with_structured_output(KetQuaDanhGia)

def node_worker(state: TrangThaiSanXuat) -> dict:
    """
    Worker thực hiện nhiệm vụ.
    Nếu đã có feedback từ lần trước, Worker biết mình cần cải thiện gì.
    """
    yeu_cau = state["yeu_cau_goc"]
    phan_hoi_cu = state.get("phan_hoi_evaluator", "")
    so_vong = state.get("so_vong_thu", 0)

    if phan_hoi_cu and so_vong > 0:
        system_prompt = f"""Bạn là một chuyên gia thực hiện nhiệm vụ theo yêu cầu.

Lần trước bạn đã hoàn thành nhiệm vụ nhưng chưa đạt tiêu chuẩn.
Đây là phản hồi chi tiết từ Evaluator về những điểm cần cải thiện:

PHẢN HỒI CẦN XỬ LÝ:
{phan_hoi_cu}

Hãy thực hiện lại nhiệm vụ, đảm bảo khắc phục TẤT CẢ những điểm được nêu trong phản hồi trên.
Đừng chỉ chỉnh sửa nhỏ — hãy thực sự cải thiện đáng kể so với lần trước."""

    else: # first time
        system_prompt = """Bạn là một chuyên gia thực hiện nhiệm vụ theo yêu cầu.
Hãy thực hiện nhiệm vụ với chất lượng cao nhất có thể.
Trả lời đầy đủ, có cấu trúc rõ ràng, và có giá trị thực tiễn."""

    ket_qua = llm_worker.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Nhiêm vụ: {yeu_cau}")
    ])

    return {
        "output_hien_tai": ket_qua.content,
        "so_vong_thu": so_vong + 1,
        "messages": [AIMessage(
            content=f"[Worker - Vòng {so_vong + 1}]\n{ket_qua.content}"
        )]
    }

def node_evaluator(state: TrangThaiSanXuat) -> dict:
    """
    Evaluator đánh giá output của Worker một cách khách quan.
    Không thiên vị — đây là LLM riêng, không phải Worker tự chấm.
    """
    output_can_danh_gia = state["output_hien_tai"]
    tieu_chuan = state["tieu_chuan_thanh_cong"]
    yeu_cau_goc = state["yeu_cau_goc"]
    phan_hoi_truoc = state.get("phan_hoi_evaluator", "")
    so_vong = state.get("so_vong_thu", 1)

    ghi_chu_them = ""
    if phan_hoi_truoc and so_vong >= 3:
        ghi_chu_them = f"""
LƯU Ý QUAN TRỌNG: Bạn đã cung cấp feedback {so_vong - 1} lần trước đó.
Feedback trước: {phan_hoi_truoc}
Nếu Worker vẫn mắc các lỗi tương tự, hãy xem xét đặt can_them_thong_tin_nguoi_dung=True
để yêu cầu người dùng làm rõ thêm yêu cầu."""

    system_prompt = f"""Bạn là Evaluator — chuyên gia đánh giá chất lượng công việc một cách khách quan và nghiêm khắc.

Nhiệm vụ của bạn: Đánh giá output của Worker dựa trên tiêu chuẩn đã đặt ra.

TIÊU CHUẨN THÀNH CÔNG:
{tieu_chuan}

YÊU CẦU GỐC:
{yeu_cau_goc}

Khi đánh giá, hãy xem xét:
1. Output có đáp ứng đầy đủ yêu cầu gốc không?
2. Output có đạt tiêu chuẩn đã đặt ra không?
3. Chất lượng thực tế của nội dung (chính xác, đầy đủ, có giá trị)
4. Nếu chưa đạt, phản hồi phải CỤ THỂ và HÀNH ĐỘNG ĐƯỢC — không phải nhận xét chung chung
{ghi_chu_them}"""

    ket_qua = llm_evaluator.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
OUTPUT CỦA WORKER CẦN ĐÁNH GIÁ:
{output_can_danh_gia}

Hãy đánh giá output này theo tiêu chuẩn đã nêu:""")
    ])

    return {
        "da_hoan_thanh": ket_qua.da_dat_tieu_chuan,
        "can_nguoi_dung": ket_qua.can_them_thong_tin_nguoi_dung,
        "phan_hoi_evaluator": ket_qua.phan_hoi_chi_tiet,
        "diem_tot_nhat": max(
            state.get("diem_tot_nhat", 0),
            ket_qua.diem_chat_luong
        )
    }

In [6]:
# routing & graph
def dinh_tuyen_sau_evaluator(state: TrangThaiSanXuat) -> str:
    """
    Quyết định sau khi Evaluator chạy xong:
    - Đạt tiêu chuẩn → END (trả về người dùng)
    - Cần người dùng → END (hỏi thêm)
    - Chưa đạt, quá nhiều vòng → END (dừng an toàn)
    - Chưa đạt, còn vòng → Worker (làm lại)
    """
    if state.get("da_hoan_thanh"):
        return END
    if state.get("can_nguoi_dung"):
        return END
    if state.get("so_vong_thu", 0) >= 4:
        return END
    return "worker"

graph_builder = StateGraph(TrangThaiSanXuat)
graph_builder.add_node("worker", node_worker)
graph_builder.add_node("evaluator", node_evaluator)

graph_builder.add_edge(START, "worker")
graph_builder.add_edge("worker", "evaluator")

graph_builder.add_conditional_edges(
    "evaluator",
    dinh_tuyen_sau_evaluator,
    {"worker": "worker", END: END}
)

memory = MemorySaver()
graph_we = graph_builder.compile(checkpointer=memory)

In [7]:
def chay_worker_evaluator(
    yeu_cau: str,
    tieu_chuan: str,
    thread_id: str = "we_test_01"
) -> str:
    """
    Chạy Worker-Evaluator workflow và trả về output cuối cùng.
    """
    config = {"configurable": {"thread_id": thread_id}}

    state_ban_dau = {
        "messages": [HumanMessage(content=yeu_cau)],
        "yeu_cau_goc": yeu_cau,
        "tieu_chuan_thanh_cong": tieu_chuan,
        "output_hien_tai": "",
        "phan_hoi_evaluator": "",
        "da_hoan_thanh": False,
        "can_nguoi_dung": False,
        "so_vong_thu": 0,
        "diem_tot_nhat": 0
    }


    print(f"Yêu cầu: {yeu_cau[:80]}...")
    print(f"Tiêu chuẩn: {tieu_chuan}")

    ket_qua = graph_we.invoke(state_ban_dau, config=config)
    
    print(f"  Số vòng đã chạy: {ket_qua['so_vong_thu']}")
    print(f"  Điểm tốt nhất: {ket_qua['diem_tot_nhat']}/10")
    print(f"  Hoàn thành: {'✅' if ket_qua['da_hoan_thanh'] else '⚠️ Dừng sau giới hạn'}")
    if ket_qua['can_nguoi_dung']:
        print(f"  ⚠️ Cần thêm thông tin từ người dùng")

    print(f"\nOUTPUT CUỐI CÙNG:")
    print(ket_qua["output_hien_tai"])

    return ket_qua["output_hien_tai"]

In [8]:
chay_worker_evaluator(
    yeu_cau="""Viết email marketing cho chiến dịch ra mắt sản phẩm trà sữa
    mới của thương hiệu 'Trà Vàng' nhắm vào Gen Z tại Hà Nội.
    Sản phẩm: Trà Sữa Oolong Mật Ong, giá 45.000đ, ra mắt ngày 15/4.""",

    tieu_chuan="""Email phải đạt TẤT CẢ các tiêu chí sau:
    1. Tiêu đề hấp dẫn, dưới 50 ký tự, tạo FOMO hoặc tò mò
    2. Có câu mở đầu kéo người đọc vào trong 2 giây đầu
    3. Nêu rõ USP của sản phẩm trong 1-2 câu
    4. Có CTA (Call to Action) rõ ràng và cụ thể
    5. Giọng văn phù hợp Gen Z: thân thiện, năng động, có thể dùng từ lóng nhẹ
    6. Độ dài phù hợp: 150-250 từ"""
)

Yêu cầu: Viết email marketing cho chiến dịch ra mắt sản phẩm trà sữa
    mới của thương h...
Tiêu chuẩn: Email phải đạt TẤT CẢ các tiêu chí sau:
    1. Tiêu đề hấp dẫn, dưới 50 ký tự, tạo FOMO hoặc tò mò
    2. Có câu mở đầu kéo người đọc vào trong 2 giây đầu
    3. Nêu rõ USP của sản phẩm trong 1-2 câu
    4. Có CTA (Call to Action) rõ ràng và cụ thể
    5. Giọng văn phù hợp Gen Z: thân thiện, năng động, có thể dùng từ lóng nhẹ
    6. Độ dài phù hợp: 150-250 từ
  Số vòng đã chạy: 3
  Điểm tốt nhất: 10/10
  Hoàn thành: ✅

OUTPUT CUỐI CÙNG:
**Tiêu đề:** 🚀 Khám Phá Trà Sữa Oolong Mật Ong - Chỉ có tại Trà Vàng từ 15/4! Đừng bỏ lỡ! 

---

**Chào bạn,**

Bạn đã sẵn sàng cho một trải nghiệm trà sữa chưa từng có? 😍 **Trà Sữa Oolong Mật Ong** sắp cập bến và hứa hẹn sẽ khiến bạn "nghiền" ngay từ lần thử đầu tiên! Chỉ **45.000đ** cho một ly trà sữa độc quyền chỉ có tại **Trà Vàng**! 

**Tại sao Trà Sữa Oolong Mật Ong lại đặc biệt?** 🍯✨  
Hãy tưởng tượng hương vị thơm ngon của trà Oolong hòa quyệ

'**Tiêu đề:** 🚀 Khám Phá Trà Sữa Oolong Mật Ong - Chỉ có tại Trà Vàng từ 15/4! Đừng bỏ lỡ! \n\n---\n\n**Chào bạn,**\n\nBạn đã sẵn sàng cho một trải nghiệm trà sữa chưa từng có? 😍 **Trà Sữa Oolong Mật Ong** sắp cập bến và hứa hẹn sẽ khiến bạn "nghiền" ngay từ lần thử đầu tiên! Chỉ **45.000đ** cho một ly trà sữa độc quyền chỉ có tại **Trà Vàng**! \n\n**Tại sao Trà Sữa Oolong Mật Ong lại đặc biệt?** 🍯✨  \nHãy tưởng tượng hương vị thơm ngon của trà Oolong hòa quyện với vị ngọt tự nhiên của mật ong, mang đến cho bạn một cảm giác thỏa mãn không thể cưỡng lại. Đây không chỉ là trà sữa, mà là một tác phẩm nghệ thuật trong từng ngụm! \n\n**Đừng bỏ lỡ cơ hội này!** ⏳  \nSản phẩm sẽ chính thức ra mắt vào **15/4** và số lượng có hạn! Hãy là một trong những người đầu tiên trải nghiệm hương vị tuyệt vời này. Bạn có thể đặt hàng ngay hôm nay để đảm bảo bạn không bỏ lỡ!\n\n👉 **Nhấn vào đây để đặt hàng ngay!** [Đặt hàng ngay]\n\nCòn chờ gì nữa? Hãy cùng bạn bè khám phá hương vị độc đáo này và trở thành